In [ ]:
#things to toggle on the interface

#parametrize:
# the room dimensions and wall curvature
# gravity
# fluid type and density
# shape of the ball
# material properties of the ball
# initial conditions



## Ball Bouncing

My first step for implementing this type of problem was to aproach it from the pure physics engine standpoint. Given python is a OOP language, it makes sense to first define a set of classes for the various features of the problem. Using pydantic, I defined a Ball class and a Environment class. Each contains various validators and physical components of the objects they represent 


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Literal
from pydantic import BaseModel, Field, computed_field, ConfigDict

class Ball(BaseModel):
    """a 3d ball with material and state"""
    model_config = ConfigDict(arbitrary_types_allowed=True)

    radius: float
    density: float
    color: str = "blue"
    elasticity: float
    max_deformation: float = 0.1
    stiffness: float = 500.0
    dilation: float = 0.0

    start_position: list[float]
    start_velocity: list[float]
    current_position: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    current_velocity: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    angular_velocity: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    acceleration: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    force: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    torque: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))

    @computed_field
    @property
    def mass(self) -> float:
        return (4.0 / 3.0) * np.pi * (self.radius ** 3) * self.density

    @computed_field
    @property
    def moment_of_inertia(self) -> float:
        return (2.0 / 5.0) * self.mass * (self.radius ** 2)

    def initialize_state(self) -> None:
        self.current_position = np.array(self.start_position, dtype=float)
        self.current_velocity = np.array(self.start_velocity, dtype=float)
        self.angular_velocity = np.zeros(3, dtype=float)
        self.acceleration = np.zeros(3, dtype=float)
        self.force = np.zeros(3, dtype=float)
        self.torque = np.zeros(3, dtype=float)
        self.dilation = 0.0


medium_types = Literal["air", "helium", "nitrogen"]
class BounceEnvironment(BaseModel):
    """environment parameters for the room"""
    model_config = ConfigDict(arbitrary_types_allowed=True)

    room_dimensions: list[float]
    gravity: float
    gravity_vector: list[float] = [0.0, 0.0, -1.0]
    linear_drag: float = 0.0
    fluid_density: float = 1.225
    drag_coefficient: float = 0.47
    medium_type: medium_types = "air"
    wall_restitution: float = 0.9
    wall_curvature: float = 0.0


Next, I defined a "Simulation" class - this is where the interactions between the an array of the Ball class and the environment are included. We include class methods for simulating the ball interactions, the ball to wall interactions, and the general environment acting on the balls (e.g., the media the balls travel through, the forces the balls experience, etc.)

There are a few limitations to this:
1. Doing this type of stepwise approach can be collisions inconsistent and wrong in behavior. Since the steps are integers, it could be that at the moment of the next "step" we have already had two balls collide, and the analysis then shows the balls at an intersected point, and redirects them with that in the state memory. That is incorrect, and could lead to incorrect "bouncing" behaviors. 
2. I used a analytical solution for the velocity and acceleration here, which could "blow up" and lead to incorrect behaviors like balls rapidly gaining momentum due to the modeled in energy leak. There are some alternatives to this that are symplectic, but I stuck for now with the pure analytical solution. 


In [3]:

class Simulation(BaseModel):
    """run a physics simulation for multiple balls"""
    model_config = ConfigDict(arbitrary_types_allowed=True)

    balls: list[Ball]
    environment: BounceEnvironment
    time_step: float
    total_time_steps: int
    positions: list[np.ndarray] = Field(default_factory=list)
    velocities: list[np.ndarray] = Field(default_factory=list)
    occupancy_grid: np.ndarray | None = None
    occupancy_history: list[np.ndarray] = Field(default_factory=list)

    def _gravity_vector(self) -> np.ndarray:
        g_dir = np.array(self.environment.gravity_vector, dtype=float)
        g_norm = np.linalg.norm(g_dir)
        if g_norm == 0.0:
            return np.zeros(3, dtype=float)
        return (g_dir / g_norm) * self.environment.gravity

    def calculate_forces(self) -> None:
        g_vec = self._gravity_vector()
        env = self.environment
        for ball in self.balls:
            ball.force = ball.mass * g_vec
            ball.torque = np.zeros(3, dtype=float)
            v = ball.current_velocity
            speed = np.linalg.norm(v)
            omega = ball.angular_velocity
            omega_mag = np.linalg.norm(omega)

            # linear drag
            if env.linear_drag != 0.0:
                ball.force += -env.linear_drag * v

            # quadratic drag
            if speed > 0.0 and env.fluid_density > 0.0:
                area = np.pi * (ball.radius ** 2)
                quad_mag = 0.5 * env.fluid_density * env.drag_coefficient * area * speed
                ball.force += -quad_mag * v

            # magnus force: lift from spin in fluid
            if speed > 0.0 and omega_mag > 1e-10 and env.fluid_density > 0.0:
                c_magnus = 0.5
                ball.force += c_magnus * env.fluid_density * (ball.radius ** 3) * np.cross(omega, v)

            # angular drag torque: viscous resistance to spin
            if omega_mag > 1e-10 and env.fluid_density > 0.0:
                c_ang_drag = 0.1
                ball.torque += -c_ang_drag * env.fluid_density * (ball.radius ** 5) * omega_mag * omega

    def update_acceleration(self) -> None:
        for ball in self.balls:
            ball.acceleration = ball.force / ball.mass

    def update_velocity(self) -> None:
        for ball in self.balls:
            ball.current_velocity = (
                ball.current_velocity + ball.acceleration * self.time_step
            )
            ball.angular_velocity = (
                ball.angular_velocity + (ball.torque / ball.moment_of_inertia) * self.time_step
            )

    def update_position(self) -> None:
        for ball in self.balls:
            ball.current_position = (
                ball.current_position + ball.current_velocity * self.time_step
            )

    def update_deformation(self) -> None:
        recovery_rate = 2.0
        for ball in self.balls:
            ball.dilation = max(0.0, ball.dilation - recovery_rate * self.time_step)

    def bounce_off_walls(self, ball: Ball) -> bool:
        dims = np.array(self.environment.room_dimensions, dtype=float)
        pos = ball.current_position.copy()
        vel = ball.current_velocity.copy()
        omega = ball.angular_velocity.copy()
        r = ball.radius
        e_eff = ball.elasticity * self.environment.wall_restitution
        hit = False
        wall_friction = 0.2
        friction_applied = False

        for axis in range(3):
            min_bound = r
            max_bound = dims[axis] - r
            e_axis = np.zeros(3, dtype=float)
            e_axis[axis] = 1.0

            colliding_low = pos[axis] < min_bound and vel[axis] < 0
            colliding_high = pos[axis] > max_bound and vel[axis] > 0

            if colliding_low or colliding_high:
                # save incoming velocity for dilation and friction before modifying
                incoming_speed = abs(vel[axis])
                r_vec = -r * e_axis if colliding_low else r * e_axis

                # compute friction using incoming velocity
                if not friction_applied and ball.mass > 0 and ball.moment_of_inertia > 0:
                    v_cp = vel + np.cross(omega, r_vec)
                    v_tang = v_cp - np.dot(v_cp, e_axis) * e_axis
                    v_tang_mag = np.linalg.norm(v_tang)
                    if v_tang_mag > 1e-10:
                        j_t = -wall_friction * v_tang
                        vel += j_t / ball.mass
                        omega += np.cross(r_vec, j_t) / ball.moment_of_inertia
                        friction_applied = True

                # apply normal impulse (bounce)
                pos[axis] = min_bound if colliding_low else max_bound
                vel[axis] = -vel[axis] * e_eff
                ball.dilation = min(ball.max_deformation, ball.dilation + incoming_speed * 0.01)
                hit = True

        ball.current_position = pos
        ball.current_velocity = vel
        ball.angular_velocity = omega
        return hit

    def handle_ball_collisions(self) -> bool:
        count = len(self.balls)
        had_collision = False

        for i in range(count):
            for j in range(i + 1, count):
                b1 = self.balls[i]
                b2 = self.balls[j]
                delta = b2.current_position - b1.current_position
                dist = np.linalg.norm(delta)
                min_dist = b1.radius + b2.radius

                if dist >= min_dist:
                    continue

                had_collision = True

                if dist == 0.0:
                    normal = np.array([1.0, 0.0, 0.0], dtype=float)
                else:
                    normal = delta / dist

                overlap = min_dist - dist
                inv_m1 = 1.0 / b1.mass if b1.mass > 0.0 else 0.0
                inv_m2 = 1.0 / b2.mass if b2.mass > 0.0 else 0.0
                total_inv_mass = inv_m1 + inv_m2
                if total_inv_mass > 0.0:
                    b1.current_position -= normal * (overlap * (inv_m1 / total_inv_mass))
                    b2.current_position += normal * (overlap * (inv_m2 / total_inv_mass))

                rel_vel = b2.current_velocity - b1.current_velocity
                vel_along_normal = np.dot(rel_vel, normal)
                if vel_along_normal > 0.0:
                    continue

                # dilation based on impact velocity, not overlap
                impact_speed = abs(vel_along_normal)
                b1.dilation = min(b1.max_deformation, b1.dilation + impact_speed * 0.01)
                b2.dilation = min(b2.max_deformation, b2.dilation + impact_speed * 0.01)

                e_eff = np.sqrt(b1.elasticity * b2.elasticity)
                impulse_mag = (-(1.0 + e_eff) * vel_along_normal) / total_inv_mass
                impulse = impulse_mag * normal

                b1.current_velocity -= impulse * inv_m1
                b2.current_velocity += impulse * inv_m2

                # tangential friction impulse (produces torque on both balls)
                ball_friction = 0.3
                r1_cp = b1.radius * normal
                r2_cp = -b2.radius * normal
                v_cp = ((b1.current_velocity + np.cross(b1.angular_velocity, r1_cp))
                        - (b2.current_velocity + np.cross(b2.angular_velocity, r2_cp)))
                v_t = v_cp - np.dot(v_cp, normal) * normal
                v_t_mag = np.linalg.norm(v_t)

                if v_t_mag > 1e-10:
                    t_hat = v_t / v_t_mag
                    r1xt = np.cross(r1_cp, t_hat)
                    r2xt = np.cross(r2_cp, t_hat)
                    inv_m_eff_t = (inv_m1 + inv_m2
                                   + np.dot(r1xt, r1xt) / b1.moment_of_inertia
                                   + np.dot(r2xt, r2xt) / b2.moment_of_inertia)
                    j_t_needed = v_t_mag / inv_m_eff_t if inv_m_eff_t > 0 else 0.0
                    j_t = min(j_t_needed, ball_friction * abs(impulse_mag))
                    fric = -j_t * t_hat

                    b1.current_velocity += fric * inv_m1
                    b2.current_velocity -= fric * inv_m2
                    b1.angular_velocity += np.cross(r1_cp, fric) / b1.moment_of_inertia
                    b2.angular_velocity -= np.cross(r2_cp, fric) / b2.moment_of_inertia

        return had_collision

    def apply_rolling_friction(self) -> None:
        """friction and torque for balls resting on walls"""
        dims = np.array(self.environment.room_dimensions, dtype=float)
        wall_friction = 0.3
        contact_tol = 0.05

        for ball in self.balls:
            r = ball.radius
            pos = ball.current_position
            vel = ball.current_velocity
            omega = ball.angular_velocity

            for axis in range(3):
                in_contact = False
                n = np.zeros(3, dtype=float)

                if pos[axis] <= r + contact_tol:
                    n[axis] = 1.0
                    in_contact = True
                elif pos[axis] >= dims[axis] - r - contact_tol:
                    n[axis] = -1.0
                    in_contact = True

                if not in_contact:
                    continue

                # contact point from center toward wall
                r_vec = -r * n
                v_cp = vel + np.cross(omega, r_vec)
                v_t = v_cp - np.dot(v_cp, n) * n
                v_t_mag = np.linalg.norm(v_t)

                if v_t_mag < 1e-10:
                    continue

                # normal force pressing ball into wall
                f_normal = max(0.0, -np.dot(ball.force, n))
                if f_normal < 1e-10:
                    continue

                f_fric = wall_friction * f_normal
                impulse_mag = f_fric * self.time_step
                # cap so we don't reverse sliding direction
                impulse_mag = min(impulse_mag, ball.mass * v_t_mag * 0.5)

                t_hat = v_t / v_t_mag
                fric_impulse = -impulse_mag * t_hat

                ball.current_velocity += fric_impulse / ball.mass
                ball.angular_velocity += np.cross(r_vec, fric_impulse) / ball.moment_of_inertia

    def _snapshot(self) -> tuple[np.ndarray, np.ndarray]:
        positions = np.stack([b.current_position for b in self.balls], axis=0)
        velocities = np.stack([b.current_velocity for b in self.balls], axis=0)
        return positions, velocities

    def init_grid(self, resolution: int = 20) -> None:
        self.occupancy_grid = np.zeros((resolution, resolution, resolution), dtype=float)

    def update_occupancy(self) -> None:
        if self.occupancy_grid is None:
            return
        dims = np.array(self.environment.room_dimensions, dtype=float)
        res = self.occupancy_grid.shape[0]
        for ball in self.balls:
            idx = ((ball.current_position / dims) * (res - 1)).astype(int)
            idx = np.clip(idx, 0, res - 1)
            self.occupancy_grid[tuple(idx)] += 1.0

    def clamp_to_bounds(self) -> None:
        """position-only clamp after ball-ball separation"""
        dims = np.array(self.environment.room_dimensions, dtype=float)
        for ball in self.balls:
            r = ball.radius
            ball.current_position = np.clip(
                ball.current_position, r, dims - r,
            )

    def step(self, store: bool = True) -> None:
        self.calculate_forces()
        self.update_acceleration()
        self.update_velocity()
        self.update_position()
        self.update_deformation()
        for ball in self.balls:
            self.bounce_off_walls(ball)
        self.handle_ball_collisions()
        self.apply_rolling_friction()
        self.clamp_to_bounds()
        if store:
            positions, velocities = self._snapshot()
            self.positions.append(positions)
            self.velocities.append(velocities)
        self.update_occupancy()
        if store and self.occupancy_grid is not None:
            self.occupancy_history.append(self.occupancy_grid.sum(axis=2).copy())

    def simulate(self) -> None:
        for ball in self.balls:
            ball.initialize_state()
        positions, velocities = self._snapshot()
        self.positions = [positions]
        self.velocities = [velocities]
        self.init_grid()
        self.occupancy_history = []
        self.update_occupancy()
        if self.occupancy_grid is not None:
            self.occupancy_history.append(self.occupancy_grid.sum(axis=2).copy())
        for _ in range(self.total_time_steps):
            self.step()

    def simulate_until(self, stop_event, max_steps: int = 500_000, store_every: int = 10, max_frames: int = 3000) -> None:
        for ball in self.balls:
            ball.initialize_state()
        positions, velocities = self._snapshot()
        self.positions = [positions]
        self.velocities = [velocities]
        self.init_grid()
        self.occupancy_history = []
        self.update_occupancy()
        if self.occupancy_grid is not None:
            self.occupancy_history.append(self.occupancy_grid.sum(axis=2).copy())
        for i in range(max_steps):
            if stop_event.is_set():
                break
            store = (i % store_every == 0) and len(self.positions) < max_frames
            self.step(store=store)

In [4]:
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import ast


def _random_non_overlapping_positions(radii, room_dims, max_tries=5000):
    rng = np.random.default_rng()
    positions = []

    for r in radii:
        mins = np.array([r, r, r], dtype=float)
        maxs = np.array(room_dims, dtype=float) - r
        placed = False

        for _ in range(max_tries):
            candidate = rng.uniform(mins, maxs)
            if all(np.linalg.norm(candidate - p) >= (r + p_r)
                   for p, p_r in zip(positions, radii[:len(positions)])):
                positions.append(candidate)
                placed = True
                break
        if not placed:
            raise ValueError(f"could not place ball with radius {r} without overlap")
    return positions


def _initial_positions(radii, room_dims, layout="random", max_tries=5000):
    """generate initial positions for different layouts while avoiding overlap"""
    rng = np.random.default_rng()
    dims = np.array(room_dims, dtype=float)

    if layout == "random":
        return _random_non_overlapping_positions(radii, room_dims, max_tries=max_tries)

    positions = []
    center = dims / 2.0

    for idx, r in enumerate(radii):
        placed = False
        for _ in range(max_tries):
            if layout == "cluster_center":
                spread = dims.min() * 0.2
                candidate = center + rng.normal(scale=spread, size=3)
            elif layout == "cluster_corner":
                base = np.array([r, r, r], dtype=float)
                spread = dims.min() * 0.2
                candidate = base + rng.normal(scale=spread, size=3)
            elif layout == "line":
                t = idx / max(1, len(radii) - 1)
                candidate = np.array([
                    r + t * (dims[0] - 2 * r),
                    dims[1] * 0.5,
                    dims[2] * 0.5,
                ], dtype=float)
            elif layout == "high_drop":
                candidate = np.array([
                    rng.uniform(r, dims[0] - r),
                    rng.uniform(r, dims[1] - r),
                    dims[2] * 0.8,
                ], dtype=float)
            else:
                candidate = rng.uniform([r, r, r], dims - r)

            if np.any(candidate < r) or np.any(candidate > dims - r):
                continue

            if all(np.linalg.norm(candidate - p) >= (r + p_r)
                   for p, p_r in zip(positions, radii[:len(positions)])):
                positions.append(candidate)
                placed = True
                break
        if not placed:
            raise ValueError(f"could not place ball with radius {r} for layout {layout}")

    return positions


def _build_balls(count, radius_range, room_dims, speed, density, layout="random", elasticity=0.5, color="blue"):
    rng = np.random.default_rng()
    r_min, r_max = radius_range
    radii = rng.uniform(r_min, r_max, count)

    positions = _initial_positions(radii, room_dims, layout=layout)

    balls = []
    for i in range(count):
        r = radii[i]
        m = density * (4.0 / 3.0) * np.pi * (r ** 3)

        direction = rng.normal(size=3)
        direction_norm = np.linalg.norm(direction)
        if direction_norm == 0.0:
            direction = np.array([1.0, 0.0, 0.0], dtype=float)
            direction_norm = 1.0
        velocity = (direction / direction_norm) * speed

        balls.append(
            Ball(
                radius=r,
                mass=m,
                color=color,
                start_position=positions[i].tolist(),
                start_velocity=velocity.tolist(),
                elasticity=elasticity,
                density=density,
            )
        )
    return balls


def _compute_scalar_fields(positions, velocities, balls, env, mode="velocity"):
    values_per_frame = []

    if mode == "velocity":
        for v in velocities:
            values_per_frame.append(np.linalg.norm(v, axis=1))
    elif mode == "potential_energy":
        g_dir = np.array(env.gravity_direction, dtype=float)
        g_norm = np.linalg.norm(g_dir)
        if g_norm == 0.0:
            g_dir = np.array([0.0, 0.0, -1.0], dtype=float)
            g_norm = 1.0
        unit_g = g_dir / g_norm
        masses = np.array([b.mass for b in balls], dtype=float)

        for pos in positions:
            height = -np.dot(pos, unit_g)
            pe = masses * env.gravity * height
            values_per_frame.append(pe)
    else:
        for pos in positions:
            values_per_frame.append(np.zeros(pos.shape[0], dtype=float))

    all_vals = np.concatenate(values_per_frame) if values_per_frame else np.array([0.0])
    vmin = float(all_vals.min())
    vmax = float(all_vals.max())
    if vmax <= vmin:
        vmax = vmin + 1.0

    return values_per_frame, vmin, vmax


def _plot_with_density(sim, balls, env, room_dims, frame_stride=1, color_mode="velocity"):
    positions = sim.positions
    velocities = sim.velocities
    occupancy_history = sim.occupancy_history
    dims = np.array(room_dims, dtype=float)

    scene_config = dict(
        xaxis=dict(range=[0, dims[0]], autorange=False),
        yaxis=dict(range=[0, dims[1]], autorange=False),
        zaxis=dict(range=[0, dims[2]], autorange=False),
        aspectmode="manual",
        aspectratio=dict(x=1, y=dims[1] / dims[0], z=dims[2] / dims[0]),
    )

    radii = np.array([b.radius for b in balls], dtype=float)
    if radii.size == 0:
        marker_sizes = 6
    else:
        r_max = radii.max()
        if r_max <= 0.0:
            marker_sizes = 6
        else:
            marker_sizes = 4.0 + (radii / r_max) * 8.0

    scalar_values, vmin, vmax = _compute_scalar_fields(positions, velocities, balls, env, mode=color_mode)

    fig = make_subplots(
        rows=2,
        cols=2,
        specs=[
            [{"type": "scene", "colspan": 2}, None],
            [{"type": "xy"}, {"type": "scene"}],
        ],
        subplot_titles=(
            "3d simulation",
            "",
            "cumulative occupancy (xy plane)",
            "occupancy surface (kde-like)",
        ),
    )

    initial = positions[0]
    initial_vals = scalar_values[0]
    initial_heat = occupancy_history[0] if occupancy_history else np.zeros((10, 10))

    fig.add_trace(
        go.Scatter3d(
            x=initial[:, 0],
            y=initial[:, 1],
            z=initial[:, 2],
            mode="markers",
            marker=dict(
                size=marker_sizes,
                color=initial_vals,
                colorscale="Viridis",
                cmin=vmin,
                cmax=vmax,
                opacity=0.8,
                colorbar=dict(title=color_mode),
            ),
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Heatmap(z=initial_heat, colorscale="Hot", showscale=False),
        row=2,
        col=1,
    )

    x_idx = np.linspace(0, dims[0], initial_heat.shape[1])
    y_idx = np.linspace(0, dims[1], initial_heat.shape[0])
    xx, yy = np.meshgrid(x_idx, y_idx)

    fig.add_trace(
        go.Surface(z=initial_heat, x=xx, y=yy, colorscale="Hot", showscale=False, opacity=0.9),
        row=2,
        col=2,
    )

    frames = []
    for t in range(0, len(positions), frame_stride):
        pos = positions[t]
        vals = scalar_values[t]
        heat = occupancy_history[t] if t < len(occupancy_history) else initial_heat
        frames.append(
            go.Frame(
                data=[
                    go.Scatter3d(
                        x=pos[:, 0],
                        y=pos[:, 1],
                        z=pos[:, 2],
                        mode="markers",
                        marker=dict(
                            size=marker_sizes,
                            color=vals,
                            colorscale="Viridis",
                            cmin=vmin,
                            cmax=vmax,
                            opacity=0.8,
                        ),
                    ),
                    go.Heatmap(z=heat, colorscale="Hot", showscale=False),
                    go.Surface(z=heat, x=xx, y=yy, colorscale="Hot", showscale=False, opacity=0.9),
                ],
                name=str(t),
                traces=[0, 1, 2],
            )
        )

    fig.update_layout(
        scene=scene_config,
        uirevision="constant_view",
        margin=dict(l=0, r=0, b=0, t=30),
        updatemenus=[
            dict(
                type="buttons",
                buttons=[
                    dict(
                        label="play",
                        method="animate",
                        args=[
                            None,
                            {"frame": {"duration": 30, "redraw": True}, "fromcurrent": True},
                        ],
                    ),
                    dict(
                        label="pause",
                        method="animate",
                        args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}],
                    ),
                ],
            )
        ],
    )

    fig.frames = frames

    return fig


ball_count = widgets.IntSlider(value=50, min=1, max=1000, step=1, description="balls")
radius_range = widgets.FloatRangeSlider(
    value=[0.3, 0.8], min=0.1, max=2.0, step=0.1, description="radius range"
)
ball_density = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="density")
ball_speed = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="speed")

room_dims = widgets.Text(
    value="[10.0, 10.0, 10.0]",
    description="room [x,y,z]",
)

gravity = widgets.FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description="gravity")
restitution = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.05, description="bounciness")

material = widgets.Dropdown(
    options=["rubber", "steel", "foam"],
    value="rubber",
    description="material",
)

color_mode = widgets.Dropdown(
    options=[
        ("velocity", "velocity"),
        ("potential energy", "potential_energy"),
    ],
    value="velocity",
    description="color by",
)

init_layout = widgets.Dropdown(
    options=[
        ("random", "random"),
        ("cluster center", "cluster_center"),
        ("cluster corner", "cluster_corner"),
        ("line", "line"),
        ("high drop", "high_drop"),
    ],
    value="random",
    description="layout",
)

run_button = widgets.Button(description="generate sim")
output = widgets.Output()


def _run_simulation(_):
    output.clear_output(wait=True)
    with output:
        try:
            dims_val = ast.literal_eval(room_dims.value)
            dims = [float(v) for v in dims_val]
            if len(dims) != 3:
                raise ValueError
        except Exception:
            dims = [10.0, 10.0, 10.0]

        material_props = {
            "rubber": {"elasticity": 0.9, "color": "red"},
            "steel": {"elasticity": 0.6, "color": "gray"},
            "foam": {"elasticity": 0.8, "color": "orange"},
        }
        props = material_props[material.value]
        elasticity = props["elasticity"]
        color = props["color"]

        env = BounceEnvironment(
            room_dimensions=dims,
            gravity=gravity.value,
            gravity_direction=[0.0, 0.0, -1.0],
            linear_drag=0.0,
            quadratic_drag=0.0,
            restitution=restitution.value,
        )
        balls = _build_balls(
            ball_count.value,
            radius_range.value,
            dims,
            ball_speed.value,
            ball_density.value,
            layout=init_layout.value,
            elasticity=elasticity,
            color=color,
        )
        sim = Simulation(
            balls=balls,
            environment=env,
            time_step=0.02,
            total_time_steps=2000,
        )
        sim.simulate()
        frame_stride = max(1, len(sim.positions) // 300)
        fig = _plot_with_density(
            sim,
            balls,
            env,
            dims,
            frame_stride=frame_stride,
            color_mode=color_mode.value,
        )
        display(fig)


run_button.on_click(_run_simulation)

controls = widgets.VBox(
    [
        ball_count,
        radius_range,
        ball_density,
        ball_speed,
        room_dims,
        gravity,
        restitution,
        material,
        color_mode,
        init_layout,
        run_button,
    ]
)

ui = widgets.HBox([controls, output])
display(ui)

In [5]:
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import ast
import os


def _random_non_overlapping_positions(radii, room_dims, max_tries=5000):
    rng = np.random.default_rng()
    positions = []

    for r in radii:
        mins = np.array([r, r, r], dtype=float)
        maxs = np.array(room_dims, dtype=float) - r
        placed = False

        for _ in range(max_tries):
            candidate = rng.uniform(mins, maxs)
            if all(np.linalg.norm(candidate - p) >= (r + p_r)
                   for p, p_r in zip(positions, radii[:len(positions)])):
                positions.append(candidate)
                placed = True
                break
        if not placed:
            raise ValueError(f"could not place ball with radius {r} without overlap")
    return positions


def _initial_positions(radii, room_dims, layout="random", max_tries=5000):
    """generate initial positions for different layouts while avoiding overlap"""
    rng = np.random.default_rng()
    dims = np.array(room_dims, dtype=float)

    if layout == "random":
        return _random_non_overlapping_positions(radii, room_dims, max_tries=max_tries)

    positions = []
    center = dims / 2.0

    for idx, r in enumerate(radii):
        placed = False
        for _ in range(max_tries):
            if layout == "cluster_center":
                spread = dims.min() * 0.2
                candidate = center + rng.normal(scale=spread, size=3)
            elif layout == "cluster_corner":
                base = np.array([r, r, r], dtype=float)
                spread = dims.min() * 0.2
                candidate = base + rng.normal(scale=spread, size=3)
            elif layout == "line":
                t = idx / max(1, len(radii) - 1)
                candidate = np.array([
                    r + t * (dims[0] - 2 * r),
                    dims[1] * 0.5,
                    dims[2] * 0.5,
                ], dtype=float)
            elif layout == "high_drop":
                candidate = np.array([
                    rng.uniform(r, dims[0] - r),
                    rng.uniform(r, dims[1] - r),
                    dims[2] * 0.8,
                ], dtype=float)
            else:
                candidate = rng.uniform([r, r, r], dims - r)

            if np.any(candidate < r) or np.any(candidate > dims - r):
                continue

            if all(np.linalg.norm(candidate - p) >= (r + p_r)
                   for p, p_r in zip(positions, radii[:len(positions)])):
                positions.append(candidate)
                placed = True
                break
        if not placed:
            raise ValueError(f"could not place ball with radius {r} for layout {layout}")

    return positions


def _build_balls(
    count,
    radius_range,
    room_dims,
    speed,
    base_density,
    layout="random",
    material_pattern="uniform",
    material_props=None,
    default_material="rubber",
):
    rng = np.random.default_rng()
    r_min, r_max = radius_range
    radii = rng.uniform(r_min, r_max, count)

    positions = _initial_positions(radii, room_dims, layout=layout)
    dims = np.array(room_dims, dtype=float)

    if material_props is None:
        material_props = {
            "rubber": {"elasticity": 0.9, "color": "red", "max_deformation": 0.15},
            "steel": {"elasticity": 0.6, "color": "gray", "max_deformation": 0.02},
            "foam": {"elasticity": 0.8, "color": "orange", "max_deformation": 0.25},
        }

    material_keys = list(material_props.keys())

    balls = []
    for i in range(count):
        r = radii[i]

        pos = positions[i]
        z_frac = 0.0 if dims[2] == 0 else pos[2] / dims[2]

        if material_pattern == "uniform":
            mat_key = default_material
        elif material_pattern == "random_mix":
            mat_key = rng.choice(material_keys)
        elif material_pattern == "layer_z":
            if z_frac < 1.0 / 3.0:
                mat_key = "rubber"
            elif z_frac < 2.0 / 3.0:
                mat_key = "foam"
            else:
                mat_key = "steel"
        else:
            mat_key = default_material

        mat = material_props.get(mat_key, material_props[default_material])
        elasticity = mat.get("elasticity", 0.9)
        color = mat.get("color", "blue")
        max_deformation = mat.get("max_deformation", 0.1)
        density = base_density

        direction = rng.normal(size=3)
        direction_norm = np.linalg.norm(direction)
        if direction_norm == 0.0:
            direction = np.array([1.0, 0.0, 0.0], dtype=float)
            direction_norm = 1.0
        velocity = (direction / direction_norm) * speed

        balls.append(
            Ball(
                radius=r,
                density=density,
                color=color,
                start_position=pos.tolist(),
                start_velocity=velocity.tolist(),
                elasticity=elasticity,
                max_deformation=max_deformation,
            )
        )
    return balls


def _compute_scalar_fields(positions, velocities, balls, env, mode="velocity"):
    values_per_frame = []

    if mode == "velocity":
        for v in velocities:
            values_per_frame.append(np.linalg.norm(v, axis=1))
    elif mode == "potential_energy":
        g_dir = np.array(env.gravity_direction, dtype=float)
        g_norm = np.linalg.norm(g_dir)
        if g_norm == 0.0:
            g_dir = np.array([0.0, 0.0, -1.0], dtype=float)
            g_norm = 1.0
        unit_g = g_dir / g_norm
        masses = np.array([b.mass for b in balls], dtype=float)

        for pos in positions:
            height = -np.dot(pos, unit_g)
            pe = masses * env.gravity * height
            values_per_frame.append(pe)
    else:
        for pos in positions:
            values_per_frame.append(np.zeros(pos.shape[0], dtype=float))

    all_vals = np.concatenate(values_per_frame) if values_per_frame else np.array([0.0])
    vmin = float(all_vals.min())
    vmax = float(all_vals.max())
    if vmax <= vmin:
        vmax = vmin + 1.0

    return values_per_frame, vmin, vmax


def _plot_with_density(sim, balls, env, room_dims, frame_stride=1, color_mode="velocity"):
    positions = sim.positions
    velocities = sim.velocities
    occupancy_history = sim.occupancy_history
    dims = np.array(room_dims, dtype=float)

    scene_config = dict(
        xaxis=dict(range=[0, dims[0]], autorange=False),
        yaxis=dict(range=[0, dims[1]], autorange=False),
        zaxis=dict(range=[0, dims[2]], autorange=False),
        aspectmode="manual",
        aspectratio=dict(x=1, y=dims[1] / dims[0], z=dims[2] / dims[0]),
    )

    radii = np.array([b.radius for b in balls], dtype=float)
    if radii.size == 0:
        marker_sizes = 6
    else:
        r_max = radii.max()
        if r_max <= 0.0:
            marker_sizes = 6
        else:
            marker_sizes = 4.0 + (radii / r_max) * 8.0

    scalar_values, vmin, vmax = _compute_scalar_fields(positions, velocities, balls, env, mode=color_mode)

    initial = positions[0]
    initial_vals = scalar_values[0]
    initial_heat = occupancy_history[0] if occupancy_history else np.zeros((10, 10))
    res_y, res_x = initial_heat.shape
    heat_x = np.linspace(0, dims[0], res_x)
    heat_y = np.linspace(0, dims[1], res_y)

    # 3d simulation figure
    sim_fig = go.Figure(
        data=[
            go.Scatter3d(
                x=initial[:, 0],
                y=initial[:, 1],
                z=initial[:, 2],
                mode="markers",
                marker=dict(
                    size=marker_sizes,
                    color=initial_vals,
                    colorscale="Viridis",
                    cmin=vmin,
                    cmax=vmax,
                    opacity=0.8,
                    colorbar=dict(title=color_mode),
                ),
            ),
        ],
    )

    frames = []
    for t in range(0, len(positions), frame_stride):
        pos = positions[t]
        vals = scalar_values[t]
        frames.append(
            go.Frame(
                data=[
                    go.Scatter3d(
                        x=pos[:, 0],
                        y=pos[:, 1],
                        z=pos[:, 2],
                        mode="markers",
                        marker=dict(
                            size=marker_sizes,
                            color=vals,
                            colorscale="Viridis",
                            cmin=vmin,
                            cmax=vmax,
                            opacity=0.8,
                        ),
                    ),
                ],
                name=str(t),
                traces=[0],
            )
        )

    sim_fig.update_layout(
        scene=scene_config,
        title=dict(text="3d simulation", x=0.5, xanchor="center"),
        uirevision="constant_view",
        margin=dict(l=20, r=20, b=20, t=50),
        updatemenus=[
            dict(
                type="buttons",
                buttons=[
                    dict(
                        label="play",
                        method="animate",
                        args=[
                            None,
                            {"frame": {"duration": 30, "redraw": True}, "fromcurrent": True},
                        ],
                    ),
                    dict(
                        label="pause",
                        method="animate",
                        args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}],
                    ),
                ],
            )
        ],
    )
    sim_fig.frames = frames

    # occupancy heatmap figure
    final_heat = occupancy_history[-1] if occupancy_history else initial_heat
    occ_fig = go.Figure(
        data=[
            go.Heatmap(
                z=final_heat,
                x=heat_x,
                y=heat_y,
                colorscale="Hot",
                showscale=True,
                colorbar=dict(title="count"),
            ),
        ],
    )
    occ_fig.update_layout(
        title=dict(text="cumulative occupancy (xy plane)", x=0.5, xanchor="center"),
        xaxis=dict(title="x", range=[0, dims[0]], showgrid=False, zeroline=False, constrain="domain"),
        yaxis=dict(title="y", range=[0, dims[1]], showgrid=False, zeroline=False, scaleanchor="x", constrain="domain"),
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=50, r=50, b=50, t=50),
        height=350,
    )

    return sim_fig, occ_fig


ball_count = widgets.IntSlider(value=50, min=1, max=1000, step=1, description="balls")
radius_range = widgets.FloatRangeSlider(
    value=[0.3, 0.8], min=0.1, max=2.0, step=0.1, description="radius range"
)
ball_density = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="density")
ball_speed = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="speed")

room_dims = widgets.Text(
    value="[10.0, 10.0, 10.0]",
    description="room [x,y,z]",
)

gravity = widgets.FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description="gravity")
restitution = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.05, description="wall restitution")

material = widgets.Dropdown(
    options=["rubber", "steel", "foam"],
    value="rubber",
    description="material",
)

material_pattern = widgets.Dropdown(
    options=[
        ("uniform", "uniform"),
        ("random mix", "random_mix"),
        ("z layers", "layer_z"),
    ],
    value="uniform",
    description="material pattern",
)

color_mode = widgets.Dropdown(
    options=[
        ("velocity", "velocity"),
        ("potential energy", "potential_energy"),
    ],
    value="velocity",
    description="color by",
)

init_layout = widgets.Dropdown(
    options=[
        ("random", "random"),
        ("cluster center", "cluster_center"),
        ("cluster corner", "cluster_corner"),
        ("line", "line"),
        ("high drop", "high_drop"),
    ],
    value="random",
    description="layout",
)

save_html = widgets.Checkbox(value=False, description="save to html")
html_filename = widgets.Text(value="spheres_sim.html", description="html file")

run_button = widgets.Button(description="generate sim")
output = widgets.Output()


sim_display = widgets.Output()
occ_display = widgets.Output()
result_area = widgets.VBox([sim_display, occ_display])


def _run_simulation(_):
    sim_display.clear_output()
    occ_display.clear_output()

    try:
        dims_val = ast.literal_eval(room_dims.value)
        dims = [float(v) for v in dims_val]
        if len(dims) != 3:
            raise ValueError
    except Exception:
        dims = [10.0, 10.0, 10.0]

    material_props = {
        "rubber": {"elasticity": 0.9, "color": "red", "max_deformation": 0.15},
        "steel": {"elasticity": 0.6, "color": "gray", "max_deformation": 0.02},
        "foam": {"elasticity": 0.8, "color": "orange", "max_deformation": 0.25},
    }

    env = BounceEnvironment(
        room_dimensions=dims,
        gravity=gravity.value,
        gravity_direction=[0.0, 0.0, -1.0],
        fluid_density=1.225,
        wall_restitution=restitution.value,
    )
    balls = _build_balls(
        ball_count.value,
        radius_range.value,
        dims,
        ball_speed.value,
        ball_density.value,
        layout=init_layout.value,
        material_pattern=material_pattern.value,
        material_props=material_props,
        default_material=material.value,
    )
    sim = Simulation(
        balls=balls,
        environment=env,
        time_step=0.02,
        total_time_steps=2000,
    )
    sim.simulate()
    frame_stride = max(1, len(sim.positions) // 300)
    sim_fig, occ_fig = _plot_with_density(
        sim,
        balls,
        env,
        dims,
        frame_stride=frame_stride,
        color_mode=color_mode.value,
    )

    if save_html.value:
        with output:
            output.clear_output()
            filename = html_filename.value.strip() or "spheres_sim.html"
            path = os.path.join(os.getcwd(), filename)
            sim_fig.write_html(path, include_plotlyjs="cdn")
            print(f"saved simulation to {path}")

            base, ext = os.path.splitext(filename)
            occ_path = os.path.join(os.getcwd(), f"{base}_occupancy{ext or '.html'}")
            occ_fig.write_html(occ_path, include_plotlyjs="cdn")
            print(f"saved occupancy to {occ_path}")

    with sim_display:
        display(sim_fig)
    with occ_display:
        display(occ_fig)


run_button.on_click(_run_simulation)

controls = widgets.VBox(
    [
        ball_count,
        radius_range,
        ball_density,
        ball_speed,
        room_dims,
        gravity,
        restitution,
        material,
        material_pattern,
        color_mode,
        init_layout,
        save_html,
        html_filename,
        run_button,
    ]
)

ui = widgets.VBox([widgets.HBox([controls, output]), result_area])
display(ui)

In [ ]:
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import ast
import os
import threading


def _random_non_overlapping_positions(radii, room_dims, max_tries=15000, allow_fallback=True):
    rng = np.random.default_rng()
    positions = []
    dims = np.array(room_dims, dtype=float)

    for r in radii:
        mins = np.array([r, r, r], dtype=float)
        maxs = dims - r
        placed = False
        for _ in range(max_tries):
            candidate = rng.uniform(mins, maxs)
            if all(np.linalg.norm(candidate - p) >= (r + p_r)
                   for p, p_r in zip(positions, radii[:len(positions)])):
                positions.append(candidate)
                placed = True
                break
        if not placed and allow_fallback:
            positions.append(rng.uniform(mins, maxs))
        elif not placed:
            raise ValueError(f"could not place ball with radius {r} without overlap")
    return positions


def _initial_positions(radii, room_dims, layout="random", max_tries=5000):
    """generate initial positions for different layouts while avoiding overlap"""
    rng = np.random.default_rng()
    dims = np.array(room_dims, dtype=float)

    if layout == "random":
        return _random_non_overlapping_positions(radii, room_dims, max_tries=max_tries)

    positions = []
    center = dims / 2.0

    for idx, r in enumerate(radii):
        placed = False
        for _ in range(max_tries):
            if layout == "cluster_center":
                spread = dims.min() * 0.2
                candidate = center + rng.normal(scale=spread, size=3)
            elif layout == "cluster_corner":
                base = np.array([r, r, r], dtype=float)
                spread = dims.min() * 0.2
                candidate = base + rng.normal(scale=spread, size=3)
            elif layout == "line":
                t = idx / max(1, len(radii) - 1)
                candidate = np.array([
                    r + t * (dims[0] - 2 * r),
                    dims[1] * 0.5,
                    dims[2] * 0.5,
                ], dtype=float)
            elif layout == "high_drop":
                z_lo = dims[2] * 0.6
                z_hi = dims[2] * 0.95
                candidate = np.array([
                    rng.uniform(r, dims[0] - r),
                    rng.uniform(r, dims[1] - r),
                    rng.uniform(z_lo, z_hi),
                ], dtype=float)
            else:
                candidate = rng.uniform([r, r, r], dims - r)

            if np.any(candidate < r) or np.any(candidate > dims - r):
                continue

            if all(np.linalg.norm(candidate - p) >= (r + p_r)
                   for p, p_r in zip(positions, radii[:len(positions)])):
                positions.append(candidate)
                placed = True
                break
        if not placed:
            # fallback: place in-bounds (may overlap) so layout never fails
            positions.append(rng.uniform([r, r, r], dims - r))

    return positions


def _build_balls(
    count,
    radius_range,
    room_dims,
    speed,
    base_density,
    layout="random",
    material_pattern="uniform",
    material_props=None,
    default_material="rubber",
    bounciness_override=None,
):
    rng = np.random.default_rng()
    r_min, r_max = radius_range
    radii = rng.uniform(r_min, r_max, count)

    positions = _initial_positions(radii, room_dims, layout=layout)
    dims = np.array(room_dims, dtype=float)

    if material_props is None:
        material_props = {
            "rubber": {"elasticity": 0.9, "color": "red", "max_deformation": 0.15, "stiffness": 500.0},
            "steel": {"elasticity": 0.6, "color": "gray", "max_deformation": 0.02, "stiffness": 2000.0},
            "foam": {"elasticity": 0.8, "color": "orange", "max_deformation": 0.25, "stiffness": 100.0},
        }

    material_keys = list(material_props.keys())

    balls = []
    for i in range(count):
        r = radii[i]

        pos = positions[i]
        z_frac = 0.0 if dims[2] == 0 else pos[2] / dims[2]

        if material_pattern == "uniform":
            mat_key = default_material
        elif material_pattern == "random_mix":
            mat_key = rng.choice(material_keys)
        elif material_pattern == "layer_z":
            if z_frac < 1.0 / 3.0:
                mat_key = "rubber"
            elif z_frac < 2.0 / 3.0:
                mat_key = "foam"
            else:
                mat_key = "steel"
        else:
            mat_key = default_material

        mat = material_props.get(mat_key, material_props[default_material])
        elasticity = bounciness_override if bounciness_override is not None else mat.get("elasticity", 0.9)
        color = mat.get("color", "blue")
        max_deformation = mat.get("max_deformation", 0.1)
        stiffness = mat.get("stiffness", 500.0)
        density = base_density

        direction = rng.normal(size=3)
        direction_norm = np.linalg.norm(direction)
        if direction_norm == 0.0:
            direction = np.array([1.0, 0.0, 0.0], dtype=float)
            direction_norm = 1.0
        velocity = (direction / direction_norm) * speed

        balls.append(
            Ball(
                radius=r,
                density=density,
                color=color,
                start_position=pos.tolist(),
                start_velocity=velocity.tolist(),
                elasticity=elasticity,
                max_deformation=max_deformation,
                stiffness=stiffness,
            )
        )
    return balls


def _compute_scalar_fields(positions, velocities, balls, env, mode="velocity"):
    values_per_frame = []

    if mode == "velocity":
        for v in velocities:
            values_per_frame.append(np.linalg.norm(v, axis=1))
    elif mode == "potential_energy":
        g_dir = np.array(env.gravity_vector, dtype=float)
        g_norm = np.linalg.norm(g_dir)
        if g_norm == 0.0:
            g_dir = np.array([0.0, 0.0, -1.0], dtype=float)
            g_norm = 1.0
        unit_g = g_dir / g_norm
        masses = np.array([b.mass for b in balls], dtype=float)

        for pos in positions:
            height = -np.dot(pos, unit_g)
            pe = masses * env.gravity * height
            values_per_frame.append(pe)
    else:
        for pos in positions:
            values_per_frame.append(np.zeros(pos.shape[0], dtype=float))

    all_vals = np.concatenate(values_per_frame) if values_per_frame else np.array([0.0])
    vmin = float(all_vals.min())
    vmax = float(all_vals.max())
    if vmax <= vmin:
        vmax = vmin + 1.0

    return values_per_frame, vmin, vmax


def _make_occupancy_figure(sim, room_dims):
    dims = np.array(room_dims, dtype=float)
    occupancy_history = sim.occupancy_history
    initial_heat = occupancy_history[0] if occupancy_history else np.zeros((10, 10))
    final_heat = occupancy_history[-1] if occupancy_history else initial_heat
    res_y, res_x = initial_heat.shape
    heat_x = np.linspace(0, dims[0], res_x)
    heat_y = np.linspace(0, dims[1], res_y)
    occ_fig = go.Figure(
        data=[
            go.Heatmap(
                z=final_heat,
                x=heat_x,
                y=heat_y,
                colorscale="Hot",
                showscale=True,
                colorbar=dict(title="count"),
            ),
        ],
    )
    occ_fig.update_layout(
        title=dict(text="cumulative occupancy (xy plane)", x=0.5, xanchor="center"),
        xaxis=dict(title="x", range=[0, dims[0]], showgrid=False, zeroline=False, constrain="domain"),
        yaxis=dict(title="y", range=[0, dims[1]], showgrid=False, zeroline=False, scaleanchor="x", constrain="domain"),
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=50, r=50, b=50, t=50),
        height=350,
    )
    return occ_fig


def _plot_with_density(sim, balls, env, room_dims, frame_stride=1, color_mode="velocity"):
    positions = sim.positions
    velocities = sim.velocities
    dims = np.array(room_dims, dtype=float)

    scene_config = dict(
        xaxis=dict(range=[0, dims[0]], autorange=False),
        yaxis=dict(range=[0, dims[1]], autorange=False),
        zaxis=dict(range=[0, dims[2]], autorange=False),
        aspectmode="manual",
        aspectratio=dict(x=1, y=dims[1] / dims[0], z=dims[2] / dims[0]),
    )

    radii = np.array([b.radius for b in balls], dtype=float)
    if radii.size == 0:
        marker_sizes = 6
    else:
        r_max = radii.max()
        if r_max <= 0.0:
            marker_sizes = 6
        else:
            marker_sizes = 4.0 + (radii / r_max) * 8.0

    scalar_values, vmin, vmax = _compute_scalar_fields(positions, velocities, balls, env, mode=color_mode)

    initial = positions[0]
    initial_vals = scalar_values[0]

    # 3d simulation figure
    sim_fig = go.Figure(
        data=[
            go.Scatter3d(
                x=initial[:, 0],
                y=initial[:, 1],
                z=initial[:, 2],
                mode="markers",
                marker=dict(
                    size=marker_sizes,
                    color=initial_vals,
                    colorscale="Viridis",
                    cmin=vmin,
                    cmax=vmax,
                    opacity=0.8,
                    colorbar=dict(title=color_mode),
                ),
            ),
        ],
    )

    frames = []
    for t in range(0, len(positions), frame_stride):
        pos = positions[t]
        vals = scalar_values[t]
        frames.append(
            go.Frame(
                data=[
                    go.Scatter3d(
                        x=pos[:, 0],
                        y=pos[:, 1],
                        z=pos[:, 2],
                        mode="markers",
                        marker=dict(
                            size=marker_sizes,
                            color=vals,
                            colorscale="Viridis",
                            cmin=vmin,
                            cmax=vmax,
                            opacity=0.8,
                        ),
                    ),
                ],
                name=str(t),
                traces=[0],
            )
        )

    sim_fig.update_layout(
        scene=scene_config,
        title=dict(text="3d simulation", x=0.5, xanchor="center"),
        uirevision="constant_view",
        margin=dict(l=20, r=20, b=20, t=50),
        updatemenus=[
            dict(
                type="buttons",
                buttons=[
                    dict(
                        label="play",
                        method="animate",
                        args=[
                            None,
                            {"frame": {"duration": 30, "redraw": True}, "fromcurrent": True},
                        ],
                    ),
                    dict(
                        label="pause",
                        method="animate",
                        args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}],
                    ),
                ],
            )
        ],
    )
    sim_fig.frames = frames
    occ_fig = _make_occupancy_figure(sim, room_dims)
    return sim_fig, occ_fig


ball_count = widgets.IntSlider(value=50, min=1, max=5000, step=1, description="balls")
radius_range = widgets.FloatRangeSlider(
    value=[0.3, 0.8], min=0.1, max=2.0, step=0.1, description="radius range"
)
ball_density = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="density")
ball_speed = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="speed")

room_dims = widgets.Text(
    value="[10.0, 10.0, 10.0]",
    description="room [x,y,z]",
)

gravity = widgets.FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description="gravity")
restitution = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.05, description="wall restitution")
bounciness = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.05, description="ball bounciness")

material = widgets.Dropdown(
    options=["rubber", "steel", "foam"],
    value="rubber",
    description="material",
)

material_pattern = widgets.Dropdown(
    options=[
        ("uniform", "uniform"),
        ("random mix", "random_mix"),
        ("z layers", "layer_z"),
    ],
    value="uniform",
    description="material pattern",
)

color_mode = widgets.Dropdown(
    options=[
        ("velocity", "velocity"),
        ("potential energy", "potential_energy"),
    ],
    value="velocity",
    description="color by",
)

init_layout = widgets.Dropdown(
    options=[
        ("random", "random"),
        ("cluster center", "cluster_center"),
        ("cluster corner", "cluster_corner"),
        ("line", "line"),
        ("high drop", "high_drop"),
    ],
    value="random",
    description="layout",
)

save_html = widgets.Checkbox(value=False, description="save to html")
html_filename = widgets.Text(value="spheres_sim.html", description="html file")

run_button = widgets.Button(description="run")
pause_button = widgets.Button(description="pause")
refresh_button = widgets.Button(description="refresh display")
status_label = widgets.Label(value="")
output = widgets.Output()
sim_display = widgets.Output()

_last_sim = None
_last_dims = None
_last_balls = None
_last_env = None
_stop_event = threading.Event()
_sim_thread = None


def _run_simulation(_):
    global _sim_thread, _last_sim, _last_dims, _last_balls, _last_env
    if _sim_thread is not None and _sim_thread.is_alive():
        status_label.value = "simulation already running"
        return
    sim_display.clear_output()
    status_label.value = "running... click pause to stop"

    try:
        dims_val = ast.literal_eval(room_dims.value)
        dims = [float(v) for v in dims_val]
        if len(dims) != 3:
            raise ValueError
    except Exception:
        dims = [10.0, 10.0, 10.0]

    material_props = {
        "rubber": {"elasticity": 0.9, "color": "red", "max_deformation": 0.15, "stiffness": 500.0},
        "steel": {"elasticity": 0.6, "color": "gray", "max_deformation": 0.02, "stiffness": 2000.0},
        "foam": {"elasticity": 0.8, "color": "orange", "max_deformation": 0.25, "stiffness": 100.0},
    }

    env = BounceEnvironment(
        room_dimensions=dims,
        gravity=gravity.value,
        gravity_vector=[0.0, 0.0, -1.0],
        wall_restitution=restitution.value,
    )
    balls = _build_balls(
        ball_count.value,
        radius_range.value,
        dims,
        ball_speed.value,
        ball_density.value,
        layout=init_layout.value,
        material_pattern=material_pattern.value,
        material_props=material_props,
        default_material=material.value,
        bounciness_override=bounciness.value,
    )
    sim = Simulation(
        balls=balls,
        environment=env,
        time_step=0.02,
        total_time_steps=0,
    )
    _stop_event.clear()
    n_balls = len(balls)
    store_every = max(1, (1 if n_balls < 500 else 5 if n_balls < 2000 else 15))
    max_steps = 500_000

    def _thread_run():
        global _last_sim, _last_dims, _last_balls, _last_env
        sim.simulate_until(_stop_event, max_steps=max_steps, store_every=store_every)
        _last_sim, _last_dims, _last_balls, _last_env = sim, dims, balls, env
        status_label.value = "stopped. click refresh display to view"

    _sim_thread = threading.Thread(target=_thread_run, daemon=True)
    _sim_thread.start()


def _on_pause(_):
    _stop_event.set()
    status_label.value = "stopping... click refresh display when done"


def _on_refresh(_):
    global _last_sim, _last_dims, _last_balls, _last_env
    sim_display.clear_output()
    if _last_sim is None or _last_dims is None or _last_balls is None or _last_env is None:
        with sim_display:
            display(widgets.HTML("run a simulation first, then pause and refresh"))
        return
    sim, dims, balls, env = _last_sim, _last_dims, _last_balls, _last_env
    frame_stride = max(1, len(sim.positions) // 300)
    sim_fig, occ_fig = _plot_with_density(
        sim, balls, env, dims, frame_stride=frame_stride, color_mode=color_mode.value
    )
    if save_html.value:
        with output:
            output.clear_output()
            filename = html_filename.value.strip() or "spheres_sim.html"
            path = os.path.join(os.getcwd(), filename)
            sim_fig.write_html(path, include_plotlyjs="cdn")
            print(f"saved simulation to {path}")
            base, ext = os.path.splitext(filename)
            occ_path = os.path.join(os.getcwd(), f"{base}_occupancy{ext or '.html'}")
            occ_fig.write_html(occ_path, include_plotlyjs="cdn")
            print(f"saved occupancy to {occ_path}")
    with sim_display:
        display(sim_fig)


run_button.on_click(_run_simulation)
pause_button.on_click(_on_pause)
refresh_button.on_click(_on_refresh)

controls = widgets.VBox(
    [
        ball_count,
        radius_range,
        ball_density,
        ball_speed,
        room_dims,
        gravity,
        restitution,
        bounciness,
        material,
        material_pattern,
        color_mode,
        init_layout,
        status_label,
        run_button,
        pause_button,
        refresh_button,
        save_html,
        html_filename,
    ]
)

ui = widgets.VBox([widgets.HBox([controls, output]), sim_display])
display(ui)



In [ ]:
heatmap_display = widgets.Output()

def _show_heatmap(_):
    heatmap_display.clear_output()
    if _last_sim is None:
        with heatmap_display:
            print("Run the 3D simulation first.")
        return
    with heatmap_display:
        occ_fig = _make_occupancy_figure(_last_sim, _last_dims)
        display(occ_fig)

show_heatmap_btn = widgets.Button(description="show occupancy heatmap")
show_heatmap_btn.on_click(_show_heatmap)
display(widgets.VBox([show_heatmap_btn, heatmap_display]))

In [ ]:


#things to add potentially:
#energy/entropoy of the balls by color codes
#how much potential energy is lost in the collision

#what is the gridcell that is the most touched throughout the simulation?
#monte carlo simulation of the grid space that is most entered by the balls

In [ ]:
"""
Lightweight 3D Bouncing Ball Simulation
Direct control over physical parameters - no material presets
"""

import numpy as np
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display
from dataclasses import dataclass


@dataclass
class Ball:
    radius: float
    density: float
    position: np.ndarray
    velocity: np.ndarray
    elasticity: float

    @property
    def mass(self):
        return (4 / 3) * np.pi * (self.radius ** 3) * self.density


@dataclass
class Environment:
    room_dims: np.ndarray
    gravity: float
    wall_restitution: float


def create_balls(count, radius_range, density, speed, elasticity, room_dims, rng=None):
    """Generate balls with random positions and velocities."""
    if rng is None:
        rng = np.random.default_rng()

    r_min, r_max = radius_range
    radii = rng.uniform(r_min, r_max, count)
    balls = []

    positions = []
    for i, r in enumerate(radii):
        # Try to place without overlap
        for _ in range(1000):
            pos = rng.uniform([r, r, r], room_dims - r)
            if all(np.linalg.norm(pos - p) >= (r + radii[j])
                   for j, p in enumerate(positions)):
                positions.append(pos)
                break
        else:
            positions.append(pos)  # Place anyway if can't find non-overlapping

        # Random velocity direction
        direction = rng.normal(size=3)
        direction /= np.linalg.norm(direction) + 1e-8
        vel = direction * speed

        balls.append(Ball(
            radius=r,
            density=density,
            position=pos.copy(),
            velocity=vel,
            elasticity=elasticity,
        ))

    return balls


def simulate(balls, env, dt=0.02, steps=2000):
    """Run physics simulation, return position history."""
    positions_history = []
    velocities_history = []

    # Copy initial state
    pos = np.array([b.position for b in balls])
    vel = np.array([b.velocity for b in balls])
    radii = np.array([b.radius for b in balls])
    masses = np.array([b.mass for b in balls])
    elasticities = np.array([b.elasticity for b in balls])

    for _ in range(steps):
        positions_history.append(pos.copy())
        velocities_history.append(vel.copy())

        # Apply gravity
        vel[:, 2] -= env.gravity * dt

        # Update positions
        pos += vel * dt

        # Wall collisions - only bounce if moving toward the wall
        for axis in range(3):
            # Lower wall: only bounce if penetrating AND moving toward wall (vel < 0)
            mask_low = (pos[:, axis] < radii) & (vel[:, axis] < 0)
            if np.any(mask_low):
                pos[mask_low, axis] = radii[mask_low]
                vel[mask_low, axis] *= -elasticities[mask_low] * env.wall_restitution

            # Upper wall: only bounce if penetrating AND moving toward wall (vel > 0)
            mask_high = (pos[:, axis] > env.room_dims[axis] - radii) & (vel[:, axis] > 0)
            if np.any(mask_high):
                pos[mask_high, axis] = env.room_dims[axis] - radii[mask_high]
                vel[mask_high, axis] *= -elasticities[mask_high] * env.wall_restitution

        # Ball-ball collisions
        for i in range(len(balls)):
            for j in range(i + 1, len(balls)):
                diff = pos[i] - pos[j]
                dist = np.linalg.norm(diff)
                min_dist = radii[i] + radii[j]

                if dist < min_dist and dist > 0:
                    # Collision normal
                    normal = diff / dist

                    # Separate balls proportional to inverse mass
                    overlap = min_dist - dist
                    m1, m2 = masses[i], masses[j]
                    total_mass = m1 + m2
                    pos[i] += normal * (overlap * m2 / total_mass)
                    pos[j] -= normal * (overlap * m1 / total_mass)

                    # Elastic collision response
                    rel_vel = vel[i] - vel[j]
                    vel_along_normal = np.dot(rel_vel, normal)

                    if vel_along_normal < 0:
                        e = min(elasticities[i], elasticities[j])
                        impulse = (-(1 + e) * vel_along_normal) / (1/m1 + 1/m2)
                        vel[i] += (impulse / m1) * normal
                        vel[j] -= (impulse / m2) * normal

    return positions_history, velocities_history


def create_figure(positions, velocities, radii, room_dims, frame_stride=1):
    """Create animated 3D plotly figure."""
    # Compute velocity magnitudes for coloring
    speeds = [np.linalg.norm(v, axis=1) for v in velocities]
    all_speeds = np.concatenate(speeds)
    vmin, vmax = all_speeds.min(), all_speeds.max()
    if vmax <= vmin:
        vmax = vmin + 1

    # Marker sizes based on radii
    r_max = radii.max() if radii.size > 0 else 1
    marker_sizes = 4 + (radii / r_max) * 12

    # Initial frame
    fig = go.Figure(data=[
        go.Scatter3d(
            x=positions[0][:, 0],
            y=positions[0][:, 1],
            z=positions[0][:, 2],
            mode='markers',
            marker=dict(
                size=marker_sizes,
                color=speeds[0],
                colorscale='Viridis',
                cmin=vmin,
                cmax=vmax,
                opacity=0.85,
                colorbar=dict(title='speed'),
            ),
        )
    ])

    # Animation frames
    frames = []
    for t in range(0, len(positions), frame_stride):
        frames.append(go.Frame(
            data=[go.Scatter3d(
                x=positions[t][:, 0],
                y=positions[t][:, 1],
                z=positions[t][:, 2],
                mode='markers',
                marker=dict(
                    size=marker_sizes,
                    color=speeds[t],
                    colorscale='Viridis',
                    cmin=vmin,
                    cmax=vmax,
                    opacity=0.85,
                ),
            )],
            name=str(t),
        ))

    fig.frames = frames

    fig.update_layout(
        scene=dict(
            xaxis=dict(range=[0, room_dims[0]], title='x'),
            yaxis=dict(range=[0, room_dims[1]], title='y'),
            zaxis=dict(range=[0, room_dims[2]], title='z'),
            aspectmode='manual',
            aspectratio=dict(
                x=1,
                y=room_dims[1] / room_dims[0],
                z=room_dims[2] / room_dims[0],
            ),
        ),
        title=dict(text='3D Bouncing Balls', x=0.5),
        margin=dict(l=10, r=10, t=50, b=10),
        updatemenus=[dict(
            type='buttons',
            showactive=False,
            y=1.15,
            x=0.5,
            xanchor='center',
            buttons=[
                dict(label='▶ Play', method='animate',
                     args=[None, {'frame': {'duration': 30, 'redraw': True},
                                  'fromcurrent': True}]),
                dict(label='⏸ Pause', method='animate',
                     args=[[None], {'frame': {'duration': 0, 'redraw': False},
                                    'mode': 'immediate'}]),
            ],
        )],
        sliders=[dict(
            active=0,
            yanchor='top',
            xanchor='left',
            currentvalue=dict(prefix='Frame: ', visible=True, xanchor='center'),
            len=0.9,
            x=0.05,
            y=0,
            steps=[dict(args=[[str(t)], dict(frame=dict(duration=0, redraw=True),
                                              mode='immediate')],
                        label=str(t), method='animate')
                   for t in range(0, len(positions), frame_stride)],
        )],
    )

    return fig


# === UI Widgets ===

# Ball properties
w_count = widgets.IntSlider(value=30, min=1, max=200, description='Ball count')
w_radius = widgets.FloatRangeSlider(value=[0.2, 0.5], min=0.1, max=1.5, step=0.05,
                                     description='Radius range')
w_density = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1,
                                 description='Density')
w_elasticity = widgets.FloatSlider(value=0.85, min=0.0, max=1.0, step=0.05,
                                    description='Elasticity')
w_speed = widgets.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.5,
                               description='Initial speed')

w_room_x = widgets.FloatSlider(value=10.0, min=5.0, max=30.0, step=1.0, description='Room X')
w_room_y = widgets.FloatSlider(value=10.0, min=5.0, max=30.0, step=1.0, description='Room Y')
w_room_z = widgets.FloatSlider(value=10.0, min=5.0, max=30.0, step=1.0, description='Room Z')
w_gravity = widgets.FloatSlider(value=9.81, min=0.0, max=25.0, step=0.5,
                                 description='Gravity')
w_wall_restitution = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.05,
                                          description='Wall bounce')

w_steps = widgets.IntSlider(value=1500, min=500, max=5000, step=100,
                             description='Time steps')

w_run = widgets.Button(description='Run Simulation', button_style='primary')
w_output = widgets.Output()


def run_simulation(_):
    w_output.clear_output()

    room_dims = np.array([w_room_x.value, w_room_y.value, w_room_z.value])

    env = Environment(
        room_dims=room_dims,
        gravity=w_gravity.value,
        wall_restitution=w_wall_restitution.value,
    )

    balls = create_balls(
        count=w_count.value,
        radius_range=w_radius.value,
        density=w_density.value,
        speed=w_speed.value,
        elasticity=w_elasticity.value,
        room_dims=room_dims,
    )

    with w_output:
        print('Simulating...')

    positions, velocities = simulate(balls, env, dt=0.02, steps=w_steps.value)

    radii = np.array([b.radius for b in balls])
    frame_stride = max(1, len(positions) // 300)

    fig = create_figure(positions, velocities, radii, room_dims, frame_stride)

    with w_output:
        w_output.clear_output()
        display(fig)


w_run._click_handlers.callbacks = []
w_run.on_click(run_simulation)
ball_box = widgets.VBox([
    widgets.HTML('<b>Ball Properties</b>'),
    w_count, w_radius, w_density, w_elasticity, w_speed,
])

env_box = widgets.VBox([
    widgets.HTML('<b>Environment</b>'),
    w_room_x, w_room_y, w_room_z, w_gravity, w_wall_restitution,
])

sim_box = widgets.VBox([
    widgets.HTML('<b>Simulation</b>'),
    w_steps, w_run,
])

controls = widgets.HBox([ball_box, env_box, sim_box])
ui = widgets.VBox([controls, w_output])

display(ui)